# CatBoost: прогноз отклонения на 10–15 минут

Ноутбук использует реальные `labels_train.csv`, `traffic.csv` и план остановок. По умолчанию выбран `data/processed`; для исходных CSV задайте `DATA_SOURCE=raw` перед запуском ядра. Модель учит поправку к `cur_dev_s`, затем восстанавливает `target_delay_s` в секундах. Основная метрика — MAE; baseline — `prediction = cur_dev_s`.

Все признаки собираются общим кодом `ml/src/features_simple.py`. Он использует телеметрию только при `event_time <= T` и `receive_time <= T`, а из расписания читает только плановую геометрию. `time_fact_begin` и `target_delay_s` в признаки не входят.

Запуск: из корня репозитория или папки `ml/notebooks`. Установите `numpy`, `pandas`, `catboost` (например, `pip install -r ml/requirements.txt`). Для Kaggle загрузите весь репозиторий или его `ml/src` вместе с `data/processed` и задайте `PROJECT_ROOT`.


In [1]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool

candidates = [Path(os.environ['PROJECT_ROOT'])] if os.environ.get('PROJECT_ROOT') else [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / 'ml' / 'src' / 'features.py').is_file() and (path / 'data' / 'raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Не найден корень проекта: задайте PROJECT_ROOT с папками ml/src и data/raw')
sys.path.insert(0, str(ROOT / 'ml'))
from src.features_simple import CAT_FEATURES, FEATURE_COLUMNS, build_features

DATA_SOURCE = os.environ.get('DATA_SOURCE', 'processed').lower()
if DATA_SOURCE not in {'raw', 'processed'}:
    raise ValueError('DATA_SOURCE должен быть raw или processed')
DATA = ROOT / 'data' / DATA_SOURCE
if not DATA.is_dir():
    raise FileNotFoundError(DATA)
ARTIFACTS = ROOT / 'ml' / 'artifacts'
SUBMISSIONS = ROOT / 'data' / 'submissions'
SEED = 42
print('Project:', ROOT, '| data source:', DATA_SOURCE)


Project: C:\Users\user\Documents\Coding\transit-schedule-predictor | data source: processed


## Загрузка и признаки

`labels_test.csv` используется только для дополнительной оценки на точках позже временной границы обучения. `validate_points.csv` не имеет таргета и нужен лишь для итогового CSV. В processed нет отдельного validate traffic: его исходный файл совпадает с test traffic, поэтому здесь берётся очищенный `test_traffic.csv`. Из обработанной телеметрии используется `coords_valid`, а колонки фактического времени остановок и `dev_s` в модель не попадают.


In [2]:
TRAFFIC_COLUMNS = ['tr_id', 'event_time', 'receive_time', 'location_valid', 'lon', 'lat', 'speed', 'heading']
SCHEDULE_COLUMNS = ['tt_action_item_id', 'tr_id', 'geom']

def load_feature_set(points_path, traffic_path, schedule_path):
    points = pd.read_csv(points_path, dtype={'sample_id': str})
    traffic_columns = TRAFFIC_COLUMNS + (['coords_valid'] if DATA_SOURCE == 'processed' else [])
    traffic = pd.read_csv(traffic_path, usecols=traffic_columns)
    if DATA_SOURCE == 'processed':
        traffic['location_valid'] = traffic.pop('coords_valid')
    schedule = pd.read_csv(schedule_path, usecols=SCHEDULE_COLUMNS)
    features = build_features(points, traffic, schedule)
    assert len(points) == len(features)
    assert features.columns.tolist() == FEATURE_COLUMNS
    return points, features

if DATA_SOURCE == 'processed':
    paths = {
        'train': (DATA / 'labels_train.csv', DATA / 'train_traffic.csv', DATA / 'train_schedule.csv'),
        'test': (DATA / 'labels_test.csv', DATA / 'test_traffic.csv', DATA / 'test_schedule.csv'),
        'validate': (DATA / 'validate_points.csv', DATA / 'test_traffic.csv', DATA / 'validate_schedule_plan.csv'),
    }
else:
    paths = {
        'train': (DATA / 'labels' / 'labels_train.csv', DATA / 'train' / 'traffic.csv', DATA / 'train' / 'schedule.csv'),
        'test': (DATA / 'labels' / 'labels_test.csv', DATA / 'test' / 'traffic.csv', DATA / 'test' / 'schedule.csv'),
        'validate': (DATA / 'validate' / 'points.csv', DATA / 'validate' / 'traffic.csv', DATA / 'validate' / 'schedule_plan.csv'),
    }
train_points, train_features = load_feature_set(*paths['train'])
test_points, test_features = load_feature_set(*paths['test'])
validate_points, validate_features = load_feature_set(*paths['validate'])
print(f'Train: {len(train_points)}, labeled test: {len(test_points)}, validate: {len(validate_points)}')
print('Доля строк с GPS на train:', round(train_features['last_lon'].notna().mean(), 3))


Train: 4434, labeled test: 353, validate: 151
Доля строк с GPS на train: 1.0


In [3]:
for name, points in [('train', train_points), ('test', test_points), ('validate', validate_points)]:
    horizon = (
        pd.to_datetime(points['target_time_begin'], format='mixed')
        - pd.to_datetime(points['T'], format='mixed')
    ).dt.total_seconds()
    assert horizon.between(600, 900, inclusive='right').all(), f'Неверный горизонт в {name}'
    assert points['sample_id'].is_unique, f'Повтор sample_id в {name}'
assert not set(train_points['sample_id']) & set(test_points['sample_id'])
assert not set(train_points['sample_id']) & set(validate_points['sample_id'])
assert 'target_delay_s' not in train_features.columns
assert 'time_fact_begin' not in train_features.columns
print('Проверки горизонта, идентификаторов и состава признаков пройдены.')


Проверки горизонта, идентификаторов и состава признаков пройдены.


## Временная проверка

Последние 20% точек по `T` идут на validation. Между обучением и проверкой оставлен 15 минутный интервал, чтобы целевые остановки train не пересекали момент начала validation. Строки не перемешиваются случайно.


In [4]:
train_time = pd.to_datetime(train_points['T'], format='mixed')
cutoff = train_time.sort_values().iloc[int(0.8 * len(train_time))]
fit_mask = (train_time < cutoff - pd.Timedelta(minutes=15)).to_numpy()
val_mask = (train_time >= cutoff).to_numpy()
assert fit_mask.sum() > 0 and val_mask.sum() > 0
print(f'Граница: {cutoff}; train={fit_mask.sum()}, validation={val_mask.sum()}')

y_train = train_points['target_delay_s'].to_numpy(dtype=float)
cur_train = train_points['cur_dev_s'].to_numpy(dtype=float)
train_pool = Pool(train_features.loc[fit_mask, FEATURE_COLUMNS], y_train[fit_mask] - cur_train[fit_mask], cat_features=CAT_FEATURES)
val_pool = Pool(train_features.loc[val_mask, FEATURE_COLUMNS], y_train[val_mask] - cur_train[val_mask], cat_features=CAT_FEATURES)


Граница: 2026-01-06 17:40:00; train=3475, validation=902


In [5]:
params = dict(
    iterations=1200,
    learning_rate=0.03,
    depth=6,
    loss_function='MAE',
    eval_metric='MAE',
    random_seed=SEED,
    task_type='CPU',
    allow_writing_files=False,
)
model = CatBoostRegressor(**params)
model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=80, use_best_model=True, verbose=200)
best_iterations = max(1, model.best_iteration_ + 1)
print('Выбрано деревьев:', best_iterations)


0:	learn: 90.0675069	test: 100.7875490	best: 100.7875490 (0)	total: 66.5ms	remaining: 1m 19s
Stopped by overfitting detector  (80 iterations wait)

bestTest = 93.99912048
bestIteration = 114

Shrink model to first 115 iterations.
Выбрано деревьев: 115


In [6]:
def mae(actual, prediction):
    return float(np.mean(np.abs(np.asarray(actual, dtype=float) - np.asarray(prediction, dtype=float))))

val_true = y_train[val_mask]
val_baseline = cur_train[val_mask]
val_prediction = val_baseline + model.predict(train_features.loc[val_mask, FEATURE_COLUMNS])
baseline_mae = mae(val_true, val_baseline)
model_mae = mae(val_true, val_prediction)
use_model = model_mae < baseline_mae
print(pd.DataFrame({'MAE, с': [mae(val_true, np.zeros_like(val_true)), baseline_mae, model_mae]},
                   index=['Нулевой прогноз', 'cur_dev_s', 'CatBoost residual']))
print('Победитель на временной проверке:', 'CatBoost' if use_model else 'cur_dev_s')

# Только точки labels_test после cutoff: для них обучение не использовало будущие метки.
test_time = pd.to_datetime(test_points['T'], format='mixed')
late_test = (test_time >= cutoff).to_numpy()
if late_test.any():
    test_true = test_points.loc[late_test, 'target_delay_s'].to_numpy(dtype=float)
    test_baseline = test_points.loc[late_test, 'cur_dev_s'].to_numpy(dtype=float)
    test_prediction = test_baseline + model.predict(test_features.loc[late_test, FEATURE_COLUMNS])
    print(f'Поздний labeled test ({late_test.sum()} точек): baseline MAE={mae(test_true, test_baseline):.2f}, '
          f'CatBoost MAE={mae(test_true, test_prediction):.2f}')


                       MAE, с
Нулевой прогноз    125.833703
cur_dev_s          101.035477
CatBoost residual   93.999121
Победитель на временной проверке: CatBoost
Поздний labeled test (75 точек): baseline MAE=86.11, CatBoost MAE=81.73


In [7]:
importance = pd.DataFrame({'feature': FEATURE_COLUMNS, 'importance': model.get_feature_importance()})
print(importance.sort_values('importance', ascending=False).head(15).to_string(index=False))


             feature  importance
           cur_dev_s   42.899286
            last_lat    7.767292
            last_lon    7.718470
          target_lat    6.292291
          target_lon    5.255860
distance_to_target_m    4.542583
        last_heading    3.639549
                hour    3.434987
       movement_5m_m    3.032421
       speed_mean_3m    2.580111
        speed_std_5m    2.200707
          last_speed    1.849270
       idle_ratio_5m    1.712744
   location_count_5m    1.648463
       speed_mean_1m    1.641552


## Финальная модель и submission

После выбора числа деревьев модель переобучается на всех доступных `labels_train.csv`. Выбор между CatBoost и baseline опирается только на временную validation. Для `validate` не используются метки, которых нет в наборе.


In [8]:
final_model = CatBoostRegressor(**{**params, 'iterations': best_iterations})
full_pool = Pool(train_features[FEATURE_COLUMNS], y_train - cur_train, cat_features=CAT_FEATURES)
final_model.fit(full_pool, verbose=False)

prediction = validate_points['cur_dev_s'].to_numpy(dtype=float).copy()
if use_model:
    prediction += final_model.predict(validate_features[FEATURE_COLUMNS])
assert np.isfinite(prediction).all()

submission = pd.DataFrame({'sample_id': validate_points['sample_id'], 'prediction': prediction})
sample = pd.read_csv(ROOT / 'data' / 'submissions' / 'sample_submission.csv', sep=';')
submission = sample[['sample_id']].astype({'sample_id': str}).merge(
    submission, on='sample_id', how='left', sort=False, validate='one_to_one'
)
assert submission['prediction'].notna().all()
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SUBMISSIONS.mkdir(parents=True, exist_ok=True)
suffix = '_processed' if DATA_SOURCE == 'processed' else ''
model_path = ARTIFACTS / f'catboost{suffix}.cbm'
final_model.save_model(str(model_path))
metadata = {
    'model': 'catboost', 'model_version': '1.0', 'data_source': DATA_SOURCE, 'prediction_target': 'target_delay_s',
    'target_mode': 'residual_to_cur_dev_s', 'selected_predictor': 'catboost' if use_model else 'cur_dev_s',
    'feature_columns': FEATURE_COLUMNS, 'cat_features': CAT_FEATURES, 'iterations': best_iterations,
    'random_seed': SEED, 'train_rows': len(train_points), 'validation_cutoff': cutoff.isoformat(),
    'validation_mae_baseline_s': baseline_mae, 'validation_mae_catboost_s': model_mae,
}
(ARTIFACTS / f'catboost{suffix}_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
submission_path = SUBMISSIONS / f'catboost{suffix}_submission.csv'
submission.to_csv(submission_path, sep=';', index=False)
print('Модель:', model_path)
print('Submission:', submission_path)
print(submission.head().to_string(index=False))


Модель: C:\Users\user\Documents\Coding\transit-schedule-predictor\ml\artifacts\catboost_processed.cbm
Submission: C:\Users\user\Documents\Coding\transit-schedule-predictor\data\submissions\catboost_processed_submission.csv
        sample_id  prediction
131672_1767670500  169.969390
131672_1767670800  169.687302
131672_1767671100  129.285037
131672_1767671400   75.969802
134040_1767672000   40.112758
